# YorTIB-200 — Prediction Evaluation

Evaluates baseline and sovereignty-preserving predictions against the gold benchmark.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("../data/yortib_200.json")
with open(DATA, encoding="utf-8") as f:
    data = json.load(f)
if isinstance(data, dict):
    data = data.get("data", data.get("items", []))
df = pd.DataFrame(data)
print("Rows:", len(df))
df.head()


In [ ]:
baseline_path = Path("../predictions/baseline.csv")
sovereign_path = Path("../predictions/sovereign.csv")

def read_prediction(path):
    p = pd.read_csv(path)
    for col in ["prediction","predicted_label","answer"]:
        if col in p.columns:
            return p[col].astype(str).str.strip()
    return p.iloc[:, -1].astype(str).str.strip()

baseline = read_prediction(baseline_path)
sovereign = read_prediction(sovereign_path)

assert len(baseline) == len(df)
assert len(sovereign) == len(df)

df["baseline_prediction"] = baseline.values
df["sovereign_prediction"] = sovereign.values
df["baseline_correct"] = df.baseline_prediction.eq(df.answer.astype(str).str.strip())
df["sovereign_correct"] = df.sovereign_prediction.eq(df.answer.astype(str).str.strip())

summary = pd.DataFrame({
    "condition": ["Baseline","Sovereignty-preserving"],
    "correct": [df.baseline_correct.sum(), df.sovereign_correct.sum()],
    "N": [len(df), len(df)],
    "accuracy": [df.baseline_correct.mean(), df.sovereign_correct.mean()]
})
display(summary)


In [ ]:
focus = (
    df.groupby("focus_word")[["baseline_correct","sovereign_correct"]]
      .mean()
      .reset_index()
      .rename(columns={
          "baseline_correct":"baseline_accuracy",
          "sovereign_correct":"sovereign_accuracy"
      })
)
display(focus)


In [ ]:
display(pd.crosstab(df.answer, df.sovereign_prediction))
